# Transformers for text classification: what happened on this call?

A debt collection agency makes thousands of phone calls to debtors. After every call, the agent labels by hand how the call ended:

| label | meaning |
|---|---|
| `Promise to pay` | the debtor commits to paying (part of) the balance |
| `Arrangement agreed` | the debtor and the agency agree a payment plan |
| `Refused to pay` | the debtor refuses to pay |
| `Disputed debt` | the debtor contests the debt itself |

**Can a model read the transcript of a call and tell us how it ended?** Such a model could check the agents' labels, flag calls that were probably mislabelled, and route disputes to the right team without waiting for a human to read them.

We will answer this question with a **pre-trained transformer**: a model that has already learned a lot about the English language from a large corpus of text, and that we adapt to our task.

## Roadmap

We first look at the data and at what a transformer is. Then we climb four levels, each of which adapts **more** of the model to our task than the previous one. Pay attention to what is trained at each level:

| Level | Section | What we do | What is trained | Pre-trained transformer weights | Name |
|---|---|---|---|---|---|
| 0 | 4 | use a model someone else fine-tuned for sentiment | nothing | unchanged | off-the-shelf model |
| 1 | 5 | turn every call into a vector with the transformer, fit a logistic regression on the vectors | the logistic regression | unchanged | **feature extraction** |
| 2 | 6 | same vectors, fit a small neural network on them | the small neural network | unchanged | **feature extraction** |
| 3 | 7 | train the transformer and a classification head together | everything | **updated** | **fine-tuning** |

Levels 1 and 2 use the transformer as a **feature extractor**: it turns text into numbers, and a classifier learns from those numbers, just like the classifiers of the previous tutorials learned from the columns of a table. The transformer itself does not learn anything. Only at level 3 do we **fine-tune** it: backpropagation updates the pre-trained weights too, so the transformer learns to produce vectors that are better suited to our task.

Each level costs more compute than the one before. The question we will answer in section 8 is what each level buys us.

| Section | |
|---|---|
| 1 | The problem and the data |
| 2 | Inside a transformer: a model built from scratch |
| 3 | The pre-trained transformer (DistilBERT) |
| 4-7 | Levels 0 to 3 |
| 8 | Comparing the levels, and the mistakes of the best model |
| 9 | Is the model cheating? |
| 10 | Using the model |

In [ ]:
import numpy as np
import pandas as pd
import torch  # pytorch
import torch.nn as nn
from datasets import Dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import (
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)

set_seed(42)  # makes the random parts (weight initialisation, shuffling) reproducible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using", device)

## 1. The data

Every row of `contact_history.csv` is one contact between the agency and a debtor: an email, a letter, an SMS, a visit or a phone call. Only **phone calls** have a `call_transcript_snippet`.

Emails have text too, but their `outcome` only records what happened to the email (`Email sent`, `Email opened`, `Email replied`), not what the debtor decided. Letters, SMS and visits have no text at all. So for our question we use the phone calls.

In [ ]:
contacts = pd.read_csv("../datasets/debt_collection/contact_history.csv")
print(contacts.shape)
contacts["channel"].value_counts()

In [ ]:
phone = contacts[contacts["channel"] == "Phone"]
phone.groupby("outcome", dropna=False)["call_transcript_snippet"].agg(
    calls="size", with_transcript="count"
)

Calls with outcome `No answer` have **no transcript at all**. That is deliberate: a call nobody picked up has nothing to transcribe. Voicemails do have text (the agent left a message), but no debtor speaks in them.

We keep the four outcomes where the debtor takes a position, and drop `Left voicemail` and `Spoke to debtor`.

**Question 1.** Why does it make sense to drop `Left voicemail` for this task? And `Spoke to debtor`? (Hint: read a few examples of each.)

In [ ]:
LABELS = ["Promise to pay", "Arrangement agreed", "Refused to pay", "Disputed debt"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

calls = phone[phone["outcome"].isin(LABELS)][["call_transcript_snippet", "outcome"]].rename(
    columns={"call_transcript_snippet": "text"}
)
calls["label"] = calls["outcome"].map(label2id)
calls["outcome"].value_counts()

In [ ]:
pd.set_option("display.max_colwidth", None)
calls.groupby("outcome").sample(1, random_state=0)[["outcome", "text"]]

The classes are not balanced: there are almost three times as many promises as arrangements. Accuracy alone would hide how well we do on the small classes, so we also report the **macro F1 score** (the F1 of each class, averaged with equal weight).

To keep training fast we use a stratified sample of 2,000 calls for training and 500 for testing. *Stratified* means every class keeps the same proportion in both sets.

If you have no GPU and level 3 turns out too slow on your laptop, lower `N_TRAIN` to 1000: the models will be a little worse, and the training time halves.

In [ ]:
N_TRAIN = 2000

train_df, test_df = train_test_split(
    calls, train_size=N_TRAIN, test_size=500, stratify=calls["label"], random_state=42
)
y_train = train_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()


def scores(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }


results = {}  # we collect the test scores of every level here

## 2. Inside a transformer

Before using a pre-trained transformer, let's see what one is made of, by building a (simple) one from scratch.

### From text to numbers: tokens

A transformer does not read characters, it reads **tokens**: pieces of words from a fixed vocabulary, each with an integer id. We use the tokenizer of DistilBERT, the pre-trained model we will use from section 3 on.

In [ ]:
model_name = "distilbert-base-uncased"  # model name from the huggingface hub
tokenizer = AutoTokenizer.from_pretrained(model_name)

example = train_df["text"].iloc[0]
print(example, "\n")
print(tokenizer.tokenize(example), "\n")
print(tokenizer(example)["input_ids"])

Note the special tokens the tokenizer adds: `[CLS]` (id 101) at the start and `[SEP]` (id 102) at the end.

Transformers have a maximum input length (512 tokens for DistilBERT), and the cost of attention grows with the square of the number of tokens. So how long are our transcripts, in tokens?

In [ ]:
n_tokens = pd.Series([len(tokenizer(t)["input_ids"]) for t in train_df["text"]])
n_tokens.describe()

All transcripts fit comfortably in 128 tokens, so from now on we truncate at 128.

In [ ]:
MAX_LENGTH = 128

### A transformer from scratch

Pytorch is (one of) the main frameworks to train deep learning models. It is a low-level library that gives a lot of flexibility to build custom models. The transformers library we use later is built on top of it.

In the code below, the transformer model is a class that inherits from `nn.Module`. It has an embedding layer that turns every token id into a vector, a stack of transformer blocks (self-attention followed by a feed-forward network) and a linear classification head. Its `forward` method takes the token ids and the attention mask and returns the logits: one score per label.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        assert self.head_dim * heads == embed_size, "Embedding size needs to be divisible by heads"

        # Define linear transformations for Q, K, V
        self.values = nn.Linear(embed_size, embed_size, bias=False)
        self.keys = nn.Linear(embed_size, embed_size, bias=False)
        self.queries = nn.Linear(embed_size, embed_size, bias=False)
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, values, keys, query, mask):
        N = query.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        print("attention forward pass: values shape:", values.shape)

        # Apply the linear transformations
        values = self.values(values)
        keys = self.keys(keys)
        queries = self.queries(query)

        # Split the embedding into self.heads different pieces
        values = values.reshape(N, value_len, self.heads, self.head_dim)
        keys = keys.reshape(N, key_len, self.heads, self.head_dim)
        queries = queries.reshape(N, query_len, self.heads, self.head_dim)

        print("values shape after splitting into the attention heads:", values.shape)

        print("computing the attention scores")

        # Compute the attention scores: one (query_len x key_len) matrix per text and per head
        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])
        if mask is not None:
            # mask has shape (N, key_len): reshape it to (N, 1, 1, key_len) so that
            # it lines up with the key dimension of energy, for every head and query
            energy = energy.masked_fill(mask[:, None, None, :] == 0, float("-1e20"))

        attention = torch.softmax(energy / (self.head_dim ** 0.5), dim=3)

        # Compute the weighted sum of the values
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads * self.head_dim
        )

        out = self.fc_out(out)

        print("dimensions of the final attention scores:", out.shape)
        return out


class TransformerBlock(nn.Module):
    def __init__(self, embed_size, heads, dropout, forward_expansion):
        super().__init__()
        self.attention = SelfAttention(embed_size, heads)  # embed_size is the size of the token embeddings, heads is the number of attention heads
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_size, embed_size),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, value, key, query, mask):
        print("calculating self-attention")
        attention = self.attention(value, key, query, mask)
        x = self.dropout(self.norm1(attention + query))
        forward = self.feed_forward(x)
        out = self.dropout(self.norm2(forward + x))
        return out


class Transformer(nn.Module):
    def __init__(self, embed_size, heads, num_layers, forward_expansion, dropout, vocab_size, max_length, num_classes):
        super().__init__()
        self.embed_size = embed_size
        self.word_embedding = nn.Embedding(vocab_size, embed_size)  # this initializes the embeddings of the tokens
        self.position_embedding = nn.Embedding(max_length, embed_size)

        self.layers = nn.ModuleList(
            [TransformerBlock(embed_size, heads, dropout, forward_expansion) for _ in range(num_layers)]
        )

        # Classification head with one output unit per class
        self.fc_out = nn.Linear(embed_size, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):  # here we pass the token ids and the mask tensors
        N, seq_length = x.shape  # N is the batch size, seq_length is the length of the sequence (how many tokens)
        print("input shape:", x.shape)

        # the position of each token in the sequence, needed for the positional encoding
        positions = torch.arange(0, seq_length).expand(N, seq_length).to(x.device)
        print("position tensor shape:", positions.shape)

        out = self.dropout(self.word_embedding(x) + self.position_embedding(positions))

        print("shape of output after first dropout", out.shape, " the last dimension is the embedding size of a token")

        for i, layer in enumerate(self.layers, start=1):
            print("running transformer block: ", i)
            # passing the same vectors three times seems odd, but remember that the self-attention mechanism
            # multiplies them with three different weight matrices (queries, keys and values)
            out = layer(out, out, out, mask)
            print("transformer block output shape", out.shape)
            print("transformer block ", i, " done")

        # Average pooling over the sequence length: one vector for the whole text.
        # not completely accurate as one would have to mask the padding tokens, but good enough for now
        out = out.mean(dim=1)

        print("after mean pooling shape", out.shape)

        # project the output to the classification head
        out = self.fc_out(out)
        print("final shape", out.shape)
        return out

Let's pass one transcript through a model with the same shape as DistilBERT: 6 transformer blocks, 12 attention heads and 768-dimensional token embeddings. We pad the input to 128 tokens: the `attention_mask` has a 1 for every real token and a 0 for every padding token, so that no token attends to padding.

In [ ]:
transcript = "Agent: You owe 250 euros. Debtor: I'll transfer it tomorrow morning, I promise."
tokens = tokenizer(transcript, return_tensors="pt", padding="max_length", max_length=MAX_LENGTH, truncation=True)

scratch_model = Transformer(
    embed_size=768, heads=12, num_layers=6, forward_expansion=4, dropout=0.1,
    vocab_size=tokenizer.vocab_size, max_length=MAX_LENGTH, num_classes=len(LABELS),
)

output = scratch_model(tokens["input_ids"], tokens["attention_mask"])
predicted_class = torch.argmax(output, dim=1).item()
print(f"Predicted outcome: {id2label[predicted_class]}")

Follow the shapes in the printout: 128 token ids become 128 vectors of size 768; every transformer block keeps that shape, but mixes information between the tokens through attention; mean pooling turns the 128 vectors into **one vector for the whole text**; and the head turns that vector into 4 scores, one per label.

In the attention layer it might be confusing that we first apply the linear transformations to the whole input and then split the result into heads. This is just an optimization trick to avoid a separate linear transformation per head. It is equivalent to what we have seen in class: https://ai.stackexchange.com/questions/41477/why-in-multi-head-attention-implementation-should-we-use-3-linear-layers-for-q

The prediction itself is **meaningless**: every weight of this model is random, the embeddings, the attention and the head. But the output has **gradients** attached to it (note the `grad_fn`):

In [ ]:
output

Pytorch keeps track of the operations that produced every tensor, and can therefore compute the gradient of a loss with respect to every parameter of the model (this is the autograd package). So we *could* train this model with a training loop: compute the loss on a batch of labelled calls, backpropagate, update the weights with an optimizer, repeat (see https://pytorch.org/tutorials/beginner/introyt/trainingyt.html).

But look at how many parameters there are to learn:

In [ ]:
print(f"{sum(p.numel() for p in scratch_model.parameters()):,} parameters")

**Question 2.** Do you think 2,000 labelled calls are enough to learn tens of millions of parameters, including what every word of English means? What would you need instead?

## 3. The pre-trained transformer

This is where **pre-training** comes in. [DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased) has almost exactly the architecture we just built, but its weights are not random: they were trained on a large corpus of English text (books and Wikipedia) to predict words that were hidden from it. To do that well, the model had to learn what words mean and how they relate to each other in a sentence. That knowledge is stored in its weights, and we get it for free.

We load DistilBERT **without** any classification head: just the tokenizer we already have, and the encoder.

In [ ]:
encoder = AutoModel.from_pretrained(model_name).to(device)
encoder.eval()  # we only use it for inference in sections 3-6: this switches off dropout

The output of the encoder, `last_hidden_state`, has one vector of size 768 per token, exactly like the output of the transformer blocks of our own model:

In [ ]:
tokens = tokenizer(transcript, return_tensors="pt").to(device)
with torch.no_grad():  # we do not train here, so we do not need gradients
    hidden = encoder(input_ids=tokens["input_ids"], attention_mask=tokens["attention_mask"]).last_hidden_state

print("tokens:", tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))
print("last_hidden_state shape:", tuple(hidden.shape), "= (batch size, number of tokens, embedding size)")

You can explore the architecture of DistilBERT visually with [netron](https://netron.app/?url=https://huggingface.co/onnxport/distilbert-base-uncased-onnx/blob/main/model.onnx) (an [onnx](https://onnx.ai/) export of the same model): the output node is `last_hidden_state`, with dimension `(batch_size, sequence_length, 768)`.

To classify a text we need **one** vector for the whole text, not one per token. There are two common ways to get it (this is called **pooling**):

- **`[CLS]` pooling**: take the vector of the first token, `[CLS]`. Through attention, it can see every other token of the text.
- **mean pooling**: average the vectors of all the real (non-padding) tokens. Our from-scratch model took a shortcut and averaged over all 128 positions, padding included; the code in section 5 does it properly.

We will try both in section 5.

## 4. Level 0: an off-the-shelf model

The Hugging Face hub has thousands of models that someone else has already fine-tuned. Can we skip training altogether? The most downloaded text classifier is a DistilBERT fine-tuned for **sentiment** on movie reviews. Let's see what it says about our calls.

In [ ]:
sentiment_analysis = pipeline(
    "sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=device
)
print(sentiment_analysis("This is a fantastic movie. I love it!"))

test_sentiment = [r["label"] for r in sentiment_analysis(test_df["text"].tolist(), batch_size=64)]
pd.crosstab(test_df["outcome"], pd.Series(test_sentiment, index=test_df.index, name="sentiment"))

**Question 3.** Can you recover the four outcomes from the sentiment? Which outcomes look alike to the sentiment model, and why is that a problem for the agency? What does this tell you about using a model for a task it was not trained for?

Level 0 cannot even produce our four labels, so it gets no score in our comparison. We need a model that learns *our* labels.

## 5. Level 1: feature extraction + logistic regression

Our first real model uses DistilBERT as a **feature extractor**. We pass every call through the encoder **once**, pool the token vectors into one vector per call, and obtain a table with one row per call and 768 numeric columns. From there on it is a classic classification problem, and we can use the same logistic regression as in Tutorial 1.

The encoder is not trained here: we run it without gradients, and its weights never change.

In [ ]:
@torch.no_grad()
def embed(texts, batch_size=64):
    # returns two feature matrices of shape (len(texts), 768): [CLS] pooling and mean pooling
    cls_vectors, mean_vectors = [], []
    for start in range(0, len(texts), batch_size):
        batch = tokenizer(
            texts[start:start + batch_size], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
        ).to(device)
        hidden = encoder(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).last_hidden_state

        cls_vectors.append(hidden[:, 0].cpu())  # the first token of every text is [CLS]

        mask = batch["attention_mask"].unsqueeze(-1)  # (batch, tokens, 1): 1 for real tokens, 0 for padding
        mean_vectors.append(((hidden * mask).sum(dim=1) / mask.sum(dim=1)).cpu())  # average over real tokens only
    return torch.cat(cls_vectors).numpy(), torch.cat(mean_vectors).numpy()


X_train_cls, X_train_mean = embed(train_df["text"].tolist())
X_test_cls, X_test_mean = embed(test_df["text"].tolist())
features = {"[CLS]": (X_train_cls, X_test_cls), "mean": (X_train_mean, X_test_mean)}
X_train_cls.shape

This is the most expensive step of levels 1 and 2, and we only do it once: every classifier from here to section 6 reuses these feature matrices.

In [ ]:
for pooling, (X_train, X_test) in features.items():
    logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    logreg.fit(X_train, y_train)
    results[f"1. logistic regression, {pooling} pooling"] = scores(y_test, logreg.predict(X_test))

pd.DataFrame(results).T

**Question 4.** Which pooling works better with a frozen encoder, and is the difference big enough to matter on 500 test calls? Something to think about: DistilBERT was pre-trained to predict hidden words, a task in which the `[CLS]` token plays no special role. Its vector only learns to summarise the whole text when the model is fine-tuned on a classification task, which is what level 3 does. Does the table agree with that expectation? And with both classifiers?

## 6. Level 2: feature extraction + a small neural network

Same features, a more flexible classifier: a neural network with one hidden layer of 768 units. This is the same shape as the classification head that the transformers library puts on top of DistilBERT in section 7 (`pre_classifier` and `classifier`).

The encoder is still not trained: the network learns from the very same feature matrices as the logistic regression. Only the classifier on top changed.

In [ ]:
for pooling, (X_train, X_test) in features.items():
    mlp = make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(768,), early_stopping=True, max_iter=500, random_state=42),
    )
    mlp.fit(X_train, y_train)
    results[f"2. neural network, {pooling} pooling"] = scores(y_test, mlp.predict(X_test))

pd.DataFrame(results).T

## 7. Level 3: fine-tuning

So far the transformer has been frozen: it produces the same vector for a call no matter what we want to predict. At level 3 we **fine-tune** it: we put a classification head on top of DistilBERT and train the two **together**, so that backpropagation also updates the pre-trained weights. The transformer learns to produce vectors that are useful for *our* four labels.

`AutoModelForSequenceClassification` loads DistilBERT with a new classification head on top.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(LABELS), id2label=id2label, label2id=label2id
)

The load report above is normal. The head (`pre_classifier` and `classifier`) is new: its weights are not in the downloaded checkpoint, so they start out random (`MISSING`). The `UNEXPECTED` weights belong to the head DistilBERT was pre-trained with (predicting hidden words), which we do not need.

The head takes the vector of the `[CLS]` token, passes it through one hidden layer (`pre_classifier`) and a final layer (`classifier`) with one output per label. You can see it at the bottom of the printout:

In [ ]:
print(model)

And this time, every parameter will be trained. `requires_grad` is `True` everywhere, the pre-trained DistilBERT weights included:

In [ ]:
n_params = 0
for name, param in model.named_parameters():
    n_params += param.numel()
    print(f"Parameter name: {name} | Shape: {tuple(param.shape)} | Requires grad: {param.requires_grad}")
print(f"\nTotal: {n_params:,} parameters, all of them trainable")

(If we set `requires_grad = False` on all the `distilbert.*` parameters, no gradients would flow into the encoder and only the head would learn. That would be level 2 again, just a slower way of doing it, because the frozen encoder would recompute the same vectors in every epoch.)

We train with the `Trainer` class of the transformers library. It expects the data as a `Dataset` of token ids, so we tokenize the calls, padding every text to 128 tokens.

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)


train_data = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False).map(tokenize, batched=True)
test_data = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False).map(tokenize, batched=True)
train_data[0].keys()

`input_ids` and `attention_mask` are the token ids and the mask we have seen before. (You can ignore `token_type_ids`: it marks which of two sentences a token belongs to, and DistilBERT does not use it.)

We give the `Trainer` a `compute_metrics` function so that it reports accuracy and macro F1 on the test set after every epoch, not just the loss.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return scores(labels, np.argmax(logits, axis=1))


training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    logging_steps=25,
    save_strategy="no",   # do not write checkpoints to disk
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
)

Let's train. This takes about a minute on a GPU, and 15-30 minutes on a laptop CPU depending on the machine (if you are on a GPU, you can watch it work with `nvidia-smi`).

In [ ]:
trainer.train()

In [ ]:
pred = trainer.predict(test_data)
y_pred = pred.predictions.argmax(axis=1)
results["3. fine-tuning"] = scores(y_test, y_pred)

## 8. Comparing the levels

In [ ]:
pd.DataFrame(results).T.round(3)

Look at the scores before reading on: levels 1 to 3 end up within a couple of points of each other. With 500 test calls, one call is worth 0.2 points of accuracy, and a difference of one or two points is within what a different random split would give you: rerun the notebook with another `random_state` and the ranking can change.

So on this data, **fine-tuning buys us nothing** that the frozen encoder did not already give. DistilBERT's pre-trained vectors separate a promise from a refusal without any training, because most debtors say what they mean in plain words.

That is a real result, not a failed experiment, and it is a common one: always try the cheap level first. Fine-tuning pays off when the signal is subtle, for example a debtor who hedges, or contests a debt without ever using the word, *and* there are enough labelled examples of those subtle calls for the transformer to learn from. In a dataset of mostly clear-cut calls they are too few to move the overall score.

**Question 5.** Given these scores, which level would you deploy, and why? Think about training time, the hardware you need, and the cost of running the model on every call. Would your answer change if the agency cared most about the ambiguous calls (a debtor who hedges, or contests part of the debt) rather than about the average score?

The overall scores do not tell us *which* mistakes the fine-tuned model makes. The confusion matrix does: rows are the true outcome, columns the predicted one.

In [ ]:
print(classification_report(y_test, y_pred, target_names=LABELS))
pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=[f"true: {label}" for label in LABELS],
    columns=[f"pred: {label}" for label in LABELS],
)

**Question 6.** Which two outcomes does the model confuse most? Read the misclassified transcripts below. Would *you* have labelled them the way the agent did? What does that mean for the idea of using this model to check the agents' labels?

In [ ]:
errors = test_df.assign(predicted=[id2label[i] for i in y_pred])
errors = errors[errors["predicted"] != errors["outcome"]]
errors[["outcome", "predicted", "text"]].head(10)

## 9. Is the model reading the debtor, or the agent?

A model learns whatever pattern predicts the label, including patterns we did not intend. Look at how often the **agent** names the outcome in their own words:

In [ ]:
def split_turns(text):
    # transcripts are either one turn per line, or all turns on one line
    return text.replace(" Debtor:", "\nDebtor:").replace(" Agent:", "\nAgent:").split("\n")


def turns_of(speaker, text):
    # everything one speaker ("Agent" or "Debtor") says in the call
    return " ".join(turn.split(":", 1)[1].strip() for turn in split_turns(text) if turn.startswith(speaker + ":"))


calls["agent_says_dispute"] = calls["text"].map(lambda t: turns_of("Agent", t)).str.contains("disput", case=False)
calls.groupby("outcome")["agent_says_dispute"].mean().round(3)

In about one in five disputed calls the agent literally says the word "dispute" (*"I understand you dispute the charge"*). A model can get those calls right without understanding the debtor at all. This is called **label leakage**: information about the answer ends up in the input. It makes the test score look better than the model really is, and a model meant to *check* the agent's labels should not rely on the agent's own words.

**Exercise.** Build a version of the data that keeps only what the **debtor** says, and redo level 1 on it (feature extraction is quick, so this takes a few minutes even on a CPU). Compare the per-class F1 scores with those of level 1 on the full transcripts: which class loses the most? Is that the class you expected? If you have a GPU, redo level 3 as well.

In [ ]:
print(calls["text"].iloc[0], "\n")
print(turns_of("Debtor", calls["text"].iloc[0]))

# your code here

## 10. Using the model

Finally, let's use the fine-tuned model on calls it has never seen, for example ones we write ourselves.

In [ ]:
new_calls = [
    "Agent: You owe 250 euros. Debtor: I'll transfer it tomorrow morning, I promise.",
    "Agent: You owe 250 euros. Debtor: I never signed up for this contract, I'm not paying for something I didn't order.",
    "Agent: You owe 250 euros. Debtor: I'm not paying you anything. Stop calling me.",
    "Agent: You owe 250 euros. Debtor: I can't pay it all at once. Could I do 50 a month?",
]
tokens = tokenizer(new_calls, return_tensors="pt", truncation=True, padding=True).to(device)

# disable gradient computation for inference
model.eval()
with torch.no_grad():
    outputs = model(input_ids=tokens["input_ids"], attention_mask=tokens["attention_mask"])

outputs.logits

The outputs of the model are the raw **logits**, one per label. The softmax function turns them into probabilities.

In [ ]:
probabilities = torch.softmax(outputs.logits, dim=1)
for text, probs in zip(new_calls, probabilities):
    prediction = int(probs.argmax())
    print(text)
    print(f"  -> {id2label[prediction]} (probability {probs[prediction]:.2f})\n")

The transformers library has a `pipeline` class that does the tokenization, the forward pass and the softmax for us, just like the sentiment model of level 0:

In [ ]:
call_classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)
call_classifier(new_calls)

**Question 7.** Do you agree with every prediction? Write a few calls of your own that you think will fool the model, and check.

**Question 8.** DistilBERT is not a *large language model* (LLM) like the ones behind ChatGPT: it has 66 million parameters, it produces vectors rather than text, and it runs on a laptop. An LLM could also classify these calls, if we simply asked it to in a prompt. What would be the advantages and disadvantages of that approach for the agency, compared to our fine-tuned model? Think about cost per call, speed, privacy of the debtors' data, and how you would measure its accuracy.